In [31]:
# ── Block 0 ── Imports and setup ───────────────────────────────────────────

import sys
import logging
from pathlib import Path

import torch

# Project root
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Project utilities
from utils.io_utils import load_fold
from utils.mlp_utils import (
    set_seed,
    prepare_all_fold_tensors,
    run_nested_random_search,
    print_final_results,
)

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

# Global seed
GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

14:20:24 [INFO] Device: cuda


In [32]:
# ── Block 1 ── Experiment configuration ────────────────────────────────────

CFG = {
    # Experiment
    "task": "hi",
    "dataset": "drd2",
    "fp_type": "ecfp4",
    "n_bits": 1024,

    # Outer folds from the Lo-Hi benchmark
    "outer_folds": [1, 2, 3],

    # Inner CV
    "inner_k": 2,

    # Reproducibility
    "random_state": GLOBAL_SEED,
}

In [33]:
SEARCH_SPACE = {
    # Architecture
    "n_layers":     [1, 2, 3],
    "n_nodes":      [128, 256, 512, 1024],  # aggiunto 128
    "r":            [0.5, 0.7, 1.0],        # aggiunto 0.7 intermedio
    "activation":   ["relu", "leaky_relu", "elu"],  # aggiunto elu
    "dropout":      [0.0, 0.2, 0.3, 0.5, 0.6],     # aggiunto 0.6
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    # Optimization
    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],  # aggiunto 2e-3
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],  # aggiunto 5e-3 e 1e-2
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER = 150
N_SEEDS = 3

In [34]:
# ── Block 2 ── Load Lo-Hi outer folds ──────────────────────────────────────

folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(
        CFG["task"],
        CFG["dataset"],
        fold_idx,
    )

    folds_data[fold_idx] = {
        "train": train_df,
        "test": test_df,
    }

    n_train = len(train_df)
    n_test = len(test_df)

    n_train_pos = int(train_df["value"].sum())
    n_train_neg = n_train - n_train_pos

    n_test_pos = int(test_df["value"].sum())
    n_test_neg = n_test - n_test_pos

    logger.info(
        f"Fold {fold_idx} | "
        f"train={n_train} "
        f"(pos={n_train_pos}, neg={n_train_neg}, "
        f"pos_ratio={n_train_pos / n_train:.3f}) | "
        f"test={n_test} "
        f"(pos={n_test_pos}, neg={n_test_neg}, "
        f"pos_ratio={n_test_pos / n_test:.3f})"
    )

14:20:24 [INFO] Loaded hi/drd2 fold 1: train=2385, test=1190
14:20:24 [INFO] Fold 1 | train=2385 (pos=1684, neg=701, pos_ratio=0.706) | test=1190 (pos=735, neg=455, pos_ratio=0.618)
14:20:24 [INFO] Loaded hi/drd2 fold 2: train=2381, test=1194
14:20:24 [INFO] Fold 2 | train=2381 (pos=1510, neg=871, pos_ratio=0.634) | test=1194 (pos=909, neg=285, pos_ratio=0.761)
14:20:24 [INFO] Loaded hi/drd2 fold 3: train=2384, test=1191
14:20:24 [INFO] Fold 3 | train=2384 (pos=1644, neg=740, pos_ratio=0.690) | test=1191 (pos=775, neg=416, pos_ratio=0.651)


In [35]:
# SMILES → ECFP4 → NumPy arrays → PyTorch tensors

folds_tensors = prepare_all_fold_tensors(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

14:20:24 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/ecfp4_train_1.npz
14:20:24 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/ecfp4_test_1.npz
14:20:24 [INFO] Fold 1 | X_train: (2385, 1024), X_test: (1190, 1024) | pos_weight: 0.416
14:20:24 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/ecfp4_train_2.npz
14:20:24 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/ecfp4_test_2.npz
14:20:24 [INFO] Fold 2 | X_train: (2381, 1024), X_test: (1194, 1024) | pos_weight: 0.577
14:20:24 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/ecfp4_train_3.npz
14:20:24 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/ecfp4_test_3.npz
14:20:24 [INFO] Fold 3 | X_train: (2384, 1024), X_test: (1191, 1024) | pos_weight: 0.450


In [36]:
# Run nested CV with random search 

logger.info(f"Estimated time: ~{N_ITER * 6 * len(CFG['outer_folds']) / 60:.0f} min")

fold_results = run_nested_random_search(
    cfg=CFG,
    folds_tensors=folds_tensors,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

14:20:24 [INFO] Estimated time: ~45 min
14:20:24 [INFO] 
OUTER FOLD 1
14:20:26 [INFO]   [1/150] inner PR-AUC=0.9479 (2s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
14:20:29 [INFO]   [2/150] inner PR-AUC=0.9398 (3s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
14:20:31 [INFO]   [3/150] inner PR-AUC=0.9308 (2s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
14:20:32 [INFO]   [4/150] inner PR-AUC=0.9437 (2s) | L=2 N=128 r=0.7 dropout=0.3 lr=3e-03
14:20:34 [INFO]   [5/150] inner PR-AUC=0.9370 (1s) | L=2 N=256 r=1.0 dropout=0.6 lr=3e-03
14:20:45 [INFO]   [6/150] inner PR-AUC=0.9416 (11s) | L=1 N=128 r=0.7 dropout=0.6 lr=1e-04
14:20:50 [INFO]   [7/150] inner PR-AUC=0.9418 (5s) | L=3 N=256 r=1.0 dropout=0.5 lr=1e-03
14:20:55 [INFO]   [8/150] inner PR-AUC=0.9438 (6s) | L=2 N=128 r=0.7 dropout=0.5 lr=5e-04
14:21:06 [INFO]   [9/150] inner PR-AUC=0.9456 (11s) | L=1 N=256 r=0.5 dropout=0.5 lr=1e-04
14:21:11 [INFO]   [10/150] inner PR-AUC=0.9407 (5s) | L=3 N=128 r=0.5 dropout=0.0 lr=5e-04
14:21:26 [INFO]   [11/150]